In [ ]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import time

import numpy as np

RUN_ID = "competition-real-localization-expanded-shards-v2"
LABEL_RUN_ID = "competition-real-localization-labels-v1"
EXPECTED_LABELS_MANIFEST_SHA256 = "8384d402874f47bfc4288f8df028310e0cf6dfd30958999679d369770b68c872"
SUPPORT_WHEEL_NAME = "numcodecs-0.16.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
EXPECTED_SUPPORT_WHEEL_SHA256 = "44869ef564a50aa545215c6a0d42ba5bbc34e9715523fb2336ada3d1fb2b331d"
EXPECTED_PARENT_INVENTORY_SHA256 = "55159ef0636d49fcc31eea6d5fe9c327be59c2813d6d0083d6cdfabc9f6112e1"
EXPECTED_INVENTORY_SHA256 = "a80c9028b21fdba1746bc2686dc5c64f0852e7954d1e758ba1357bcca5aa0edc"
FINAL_PROBE_STEMS = {
    "44b6_12dfb391", "44b6_267148e4", "6bba_062c8d37", "6bba_07e24132"
}
STARTED = time.monotonic()
INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
OUTPUT_ROOT = WORKING / "competition_real_localization_expanded_shards_v2"
label_dataset_candidates = [
    INPUT_ROOT / "biohub-real-localization-labels-v2",
    INPUT_ROOT / "datasets" / "indarkarhana" / "biohub-real-localization-labels-v2",
]
label_dataset_roots = [path for path in label_dataset_candidates if path.is_dir()]
if len(label_dataset_roots) != 1:
    raise RuntimeError({"eligible_label_dataset_roots": [str(path) for path in label_dataset_roots]})
DATASET_ROOT = label_dataset_roots[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".partial")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")
    temporary.replace(path)


label_root = DATASET_ROOT / "biohub_real_localization_labels_v1"
labels_manifest_path = label_root / "labels_manifest.json"
if not labels_manifest_path.is_file() or sha256_file(labels_manifest_path) != EXPECTED_LABELS_MANIFEST_SHA256:
    raise RuntimeError({"eligible_label_manifest": str(labels_manifest_path)})
inventory_candidates = [
    INPUT_ROOT / "biohub-real-localization-expanded-inventory-v2" / "expanded_inventory.json",
    INPUT_ROOT / "datasets" / "indarkarhana" / "biohub-real-localization-expanded-inventory-v2" / "expanded_inventory.json",
]
inventory_paths = [path for path in inventory_candidates if path.is_file()]
if len(inventory_paths) != 1:
    raise RuntimeError({"eligible_expanded_inventory_paths": [str(path) for path in inventory_paths]})
inventory_path = inventory_paths[0]
support_wheel = DATASET_ROOT / SUPPORT_WHEEL_NAME
if not support_wheel.is_file() or sha256_file(support_wheel) != EXPECTED_SUPPORT_WHEEL_SHA256:
    raise RuntimeError({"eligible_support_wheel": str(support_wheel)})
site_root = WORKING / "offline_site"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
    "--target", str(site_root), str(support_wheel),
], check=True)
sys.path.insert(0, str(site_root))
from numcodecs import Blosc


if (
    sha256_file(labels_manifest_path) != EXPECTED_LABELS_MANIFEST_SHA256
    or sha256_file(inventory_path) != EXPECTED_INVENTORY_SHA256
):
    raise RuntimeError("label package provenance changed")
labels_manifest = json.loads(labels_manifest_path.read_text())
inventory = json.loads(inventory_path.read_text())
movies = inventory.get("movies")
if not (
    labels_manifest.get("status") == "complete"
    and labels_manifest.get("run_id") == LABEL_RUN_ID
    and labels_manifest.get("inventory_sha256") == EXPECTED_PARENT_INVENTORY_SHA256
    and labels_manifest.get("competition_test_data_read") is False
    and labels_manifest.get("public_leaderboard_used_for_selection") is False
    and labels_manifest.get("authorized_for_submission") is False
    and inventory.get("status") == "complete"
    and inventory.get("run_id") == "competition-real-localization-expanded-inventory-v2"
    and inventory.get("parent_inventory_sha256") == EXPECTED_PARENT_INVENTORY_SHA256
    and inventory.get("expansion_policy", {}).get("roles_expanded") == ["optimization"]
    and inventory.get("expansion_policy", {}).get("selection_centers_changed") is False
    and inventory.get("expansion_policy", {}).get("sealed_audit_centers_changed") is False
    and inventory.get("competition_train_data_read") is True
    and inventory.get("competition_test_data_read") is False
    and inventory.get("public_leaderboard_used_for_selection") is False
    and inventory.get("authorized_for_submission") is False
    and isinstance(movies, list)
    and len(movies) == 115
    and not ({row["stem"] for row in movies} & FINAL_PROBE_STEMS)
):
    raise RuntimeError("ineligible train-only replay contract")
label_records = {row["stem"]: row for row in labels_manifest["files"]}
if set(label_records) != {row["stem"] for row in movies}:
    raise RuntimeError("label movie inventory changed")
for stem, row in label_records.items():
    path = label_root / row["path"]
    if path.stat().st_size != int(row["bytes"]) or sha256_file(path) != row["sha256"]:
        raise RuntimeError(f"label file changed: {stem}")

candidate_paths = [
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/train"),
    Path("/kaggle/input/biohub-cell-tracking-during-development/train"),
]
candidate_paths.extend(path / "train" for path in INPUT_ROOT.iterdir() if path.is_dir())
train_roots = []
for path in candidate_paths:
    if path.is_dir() and len(list(path.glob("*.geff"))) == 199:
        resolved = path.resolve()
        if resolved not in train_roots:
            train_roots.append(resolved)
if len(train_roots) != 1:
    raise RuntimeError({"eligible_train_roots": [str(path) for path in train_roots]})
train_root = train_roots[0]
if train_root.name != "train" or "test" in train_root.as_posix().lower():
    raise RuntimeError("competition train root boundary failed")


def open_frame(movie_root, frame):
    metadata = json.loads((movie_root / "0" / "zarr.json").read_text())
    expected_codecs = [
        {"name": "bytes", "configuration": {"endian": "little"}},
        {
            "name": "blosc",
            "configuration": {
                "typesize": 2,
                "cname": "zstd",
                "clevel": 1,
                "shuffle": "bitshuffle",
                "blocksize": 0,
            },
        },
    ]
    if not (
        metadata.get("zarr_format") == 3
        and metadata.get("node_type") == "array"
        and metadata.get("shape", [])[1:] == [64, 256, 256]
        and metadata.get("data_type") == "uint16"
        and metadata.get("chunk_grid", {}).get("configuration", {}).get("chunk_shape")
        == [1, 64, 256, 256]
        and metadata.get("chunk_key_encoding", {}).get("configuration", {}).get("separator") == "/"
        and metadata.get("codecs") == expected_codecs
        and 0 <= int(frame) < int(metadata["shape"][0])
    ):
        raise RuntimeError(f"unsupported competition Zarr contract: {movie_root.name}")
    chunk_path = movie_root / "0" / "c" / str(int(frame)) / "0" / "0" / "0"
    codec = Blosc(cname="zstd", clevel=1, shuffle=Blosc.BITSHUFFLE, blocksize=0)
    decoded = codec.decode(chunk_path.read_bytes())
    expected_bytes = 1 * 64 * 256 * 256 * np.dtype("<u2").itemsize
    if len(decoded) != expected_bytes:
        raise RuntimeError(f"decoded chunk size changed: {movie_root.name} t={frame}")
    return np.frombuffer(decoded, dtype="<u2").reshape(1, 64, 256, 256)[0]

OUTPUT_ROOT.mkdir()
records = []
frame_records = []
for movie_index, movie in enumerate(movies, start=1):
    stem = movie["stem"]
    role = movie["role"]
    with np.load(label_root / label_records[stem]["path"], allow_pickle=False) as data:
        node_ids = data["node_ids"].astype(np.int64, copy=False)
        times = data["times"].astype(np.int32, copy=False)
        coords = data["coords_voxel"].astype(np.float32, copy=False)
        edges_by_id = data["edges"].astype(np.int64, copy=False).reshape(-1, 2)
    movie_root = train_root / f"{stem}.zarr"
    frames = {}
    for frame in movie["required_frames"]:
        volume = open_frame(movie_root, int(frame))
        if volume.shape != (64, 256, 256):
            raise RuntimeError(f"unexpected frame shape: {stem} t={frame}")
        frames[int(frame)] = volume
        frame_records.append({
            "stem": stem,
            "frame": int(frame),
            "shape": list(volume.shape),
            "dtype": str(volume.dtype),
            "sha256": hashlib.sha256(np.ascontiguousarray(volume).tobytes()).hexdigest(),
        })
    for center in movie["center_frames"]:
        center = int(center)
        triplet_times = (center - 1, center, center + 1)
        selected_mask = np.isin(times, triplet_times)
        selected_ids = node_ids[selected_mask]
        selected_times = times[selected_mask]
        selected_coords = coords[selected_mask]
        if not np.any(selected_times == center):
            raise RuntimeError(f"center labels missing: {stem} t={center}")
        by_id = {int(node_id): row for row, node_id in enumerate(selected_ids)}
        edge_rows = np.asarray([
            (by_id[int(source)], by_id[int(target)])
            for source, target in edges_by_id.tolist()
            if int(source) in by_id and int(target) in by_id
        ], dtype=np.int64).reshape(-1, 2)
        outgoing = {}
        for source, target in edge_rows.tolist():
            outgoing.setdefault(int(source), []).append(int(target))
        divisions = np.asarray(sorted(
            source for source, targets in outgoing.items() if len(targets) >= 2
        ), dtype=np.int64)
        volumes = np.stack([frames[frame] for frame in triplet_times])
        pooled = np.ascontiguousarray(volumes[:, :, ::4, ::4])
        node_rows = np.column_stack((
            selected_times - center + 1,
            selected_coords,
            selected_ids,
        )).astype(np.float32)
        corrected = node_rows[:, 1:4].copy()
        corrected[:, 1:] /= 4.0
        if (
            pooled.shape != (3, 64, 64, 64)
            or np.any(corrected < 0)
            or np.any(corrected >= np.asarray(pooled.shape[1:])[None])
        ):
            raise RuntimeError(f"pooling or coordinates invalid: {stem} t={center}")
        destination = OUTPUT_ROOT / role / f"{stem}__t{center:04d}.npz"
        destination.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            destination,
            volumes=pooled,
            nodes=node_rows,
            edges=edge_rows,
            divisions=divisions,
            voxel_um_pooled=np.asarray((1.625, 1.625, 1.625), dtype=np.float32),
        )
        critical = set()
        for source, targets in outgoing.items():
            if len(targets) >= 2:
                critical.add(source)
                critical.update(targets)
        critical_center = sum(int(node_rows[row, 0]) == 1 for row in critical)
        if role in {"selection", "sealed_audit"} and critical_center <= 0:
            raise RuntimeError(f"gate shard lost division: {destination.name}")
        records.append({
            "path": destination.relative_to(OUTPUT_ROOT).as_posix(),
            "stem": stem,
            "embryo": movie["embryo"],
            "role": role,
            "center_frame": center,
            "nodes": len(node_rows),
            "center_nodes": int(np.sum(node_rows[:, 0] == 1)),
            "division_critical_center_nodes": critical_center,
            "bytes": destination.stat().st_size,
            "sha256": sha256_file(destination),
        })
    if movie_index % 20 == 0:
        print(f"built {movie_index}/{len(movies)} real movies", flush=True)

frame_digest_manifest = {
    "schema_version": 1,
    "status": "complete",
    "run_id": "competition-real-localization-kaggle-source-digests-v2",
    "inventory_sha256": EXPECTED_INVENTORY_SHA256,
    "files": frame_records,
    "summary": {"frames": len(frame_records)},
    "competition_train_data_read": True,
    "competition_test_data_read": False,
    "public_leaderboard_used_for_selection": False,
    "submission_created": False,
    "authorized_for_submission": False,
}
frame_manifest_path = OUTPUT_ROOT / "source_frame_digest_manifest.json"
atomic_json(frame_manifest_path, frame_digest_manifest)
counts = {
    role: {
        "shards": sum(row["role"] == role for row in records),
        "center_nodes": sum(row["center_nodes"] for row in records if row["role"] == role),
        "division_critical_center_nodes": sum(
            row["division_critical_center_nodes"] for row in records if row["role"] == role
        ),
    }
    for role in ("optimization", "selection", "sealed_audit")
}
if (
    len(records) != 511
    or len(frame_records) != 1527
    or counts["optimization"]["shards"] != 480
    or counts["selection"]["shards"] != 17
    or counts["sealed_audit"]["shards"] != 14
    or any(counts[role]["center_nodes"] <= 0 for role in counts)
    or any(counts[role]["division_critical_center_nodes"] <= 0 for role in ("selection", "sealed_audit"))
):
    raise RuntimeError("real localization output inventory changed")
manifest = {
    "schema_version": 1,
    "status": "complete",
    "run_id": RUN_ID,
    "source_mode": "kaggle_cpu_direct_competition_train_expanded_optimization_only",
    "inventory_sha256": EXPECTED_INVENTORY_SHA256,
    "labels_manifest_sha256": EXPECTED_LABELS_MANIFEST_SHA256,
    "frame_cache_manifest_sha256": sha256_file(frame_manifest_path),
    "geff_cache_manifest_sha256": labels_manifest["geff_cache_manifest_sha256"],
    "files": records,
    "summary": {
        "shards": len(records),
        "bytes": sum(row["bytes"] for row in records),
        "source_frames": len(frame_records),
        "by_role": counts,
    },
    "excluded_final_probe_stems": sorted(FINAL_PROBE_STEMS),
    "geometry": {
        "source_shape_zyx": [64, 256, 256],
        "pooled_shape_zyx": [64, 64, 64],
        "xy_stride": 4,
        "pooled_voxel_um": [1.625, 1.625, 1.625],
    },
    "competition_train_data_read": True,
    "competition_test_data_read": False,
    "public_leaderboard_used_for_selection": False,
    "submission_created": False,
    "authorized_for_submission": False,
}
manifest_path = OUTPUT_ROOT / "real_localization_shard_manifest.json"
atomic_json(manifest_path, manifest)
terminal = {
    "schema_version": 1,
    "status": "completed",
    "run_id": "competition-real-localization-expanded-kaggle-cache-v2",
    "elapsed_seconds": round(time.monotonic() - STARTED, 3),
    "accelerator": "cpu",
    "gpu_used": False,
    "shard_manifest_sha256": sha256_file(manifest_path),
    "shards": len(records),
    "competition_train_data_read": True,
    "competition_test_data_read": False,
    "public_leaderboard_used_for_selection": False,
    "submission_created": False,
    "authorized_for_submission": False,
}
atomic_json(WORKING / "launcher_terminal.json", terminal)
print(json.dumps(terminal, indent=2, sort_keys=True))


# Emit one immutable uncompressed tar so the external trainer can stream and
# verify hundreds of already-compressed NPZ shards without directory races.
import tarfile
archive_path = WORKING / "biohub-real-localization-expanded-shards-v2.tar"
with tarfile.open(archive_path, mode="w") as archive:
    archive.add(OUTPUT_ROOT, arcname=OUTPUT_ROOT.name, recursive=True)
archive_sha256 = sha256_file(archive_path)
(WORKING / "biohub-real-localization-expanded-shards-v2.tar.sha256").write_text(
    f"{archive_sha256}  {archive_path.name}\n"
)
terminal = json.loads((WORKING / "launcher_terminal.json").read_text())
terminal["archive_name"] = archive_path.name
terminal["archive_bytes"] = archive_path.stat().st_size
terminal["archive_sha256"] = archive_sha256
atomic_json(WORKING / "launcher_terminal.json", terminal)
print(json.dumps(terminal, indent=2, sort_keys=True))
